# Amazon Product Reviews — Text Cleaning (Week 1)

**Project:** Amazon Product Reviews Sentiment Dashboard  
**Objective:** Clean review text by removing HTML tags, punctuation, and stopwords.  
**Dataset:** [Amazon Fine Food Reviews (Kaggle)](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)  
**Author:** Shravani Kurlapkar

## 1. Import Libraries

In [ ]:
import pandas as pd
import string
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

print('All libraries loaded successfully.')

## 2. Load the Dataset

Place `Reviews.csv` in the `data/raw/` folder (downloaded from Kaggle).

In [ ]:
FILE_PATH = '../data/raw/Reviews.csv'

df = pd.read_csv(FILE_PATH)

print(f'Dataset shape : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns       : {df.columns.tolist()}')
df.head(3)

## 3. Data Exploration & Basic Cleanup

In [ ]:
# Check nulls
null_count = df['Text'].isnull().sum()
print(f'Null values in Text column: {null_count}')

# Check duplicates
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# Drop nulls and duplicates
df = df.dropna(subset=['Text']).drop_duplicates().reset_index(drop=True)
print(f'Shape after cleanup: {df.shape}')

In [ ]:
# Sample raw review to see what needs cleaning
print('--- Sample raw review ---')
print(df['Text'].iloc[0])

## 4. Define Cleaning Functions

Three cleaning steps:
1. **Remove HTML tags** — BeautifulSoup
2. **Remove punctuation** — `string.punctuation`
3. **Remove stopwords** — NLTK English stopwords

In [ ]:
def remove_html(text):
    """Strip HTML tags using BeautifulSoup."""
    return BeautifulSoup(text, 'html.parser').get_text()

def remove_punctuation(text):
    """Remove all punctuation characters."""
    return text.translate(str.maketrans('', '', string.punctuation))

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    """Lowercase and remove English stopwords."""
    words = text.lower().split()
    return ' '.join(w for w in words if w not in stop_words)

print(f'Loaded {len(stop_words)} English stopwords.')
print('Cleaning functions defined.')

## 5. Apply the Cleaning Pipeline

In [ ]:
df['Text'] = df['Text'].astype(str)

print('Step 1/3 — Removing HTML tags...')
df['cleaned_text'] = df['Text'].apply(remove_html)

print('Step 2/3 — Removing punctuation...')
df['cleaned_text'] = df['cleaned_text'].apply(remove_punctuation)

print('Step 3/3 — Removing stopwords...')
df['cleaned_text'] = df['cleaned_text'].apply(remove_stopwords)

print('Cleaning complete!')

## 6. Verify — Before vs After

In [ ]:
for i in range(5):
    print(f'=== Review {i+1} ===')
    print(f'BEFORE: {df["Text"].iloc[i][:300]}')
    print(f'AFTER : {df["cleaned_text"].iloc[i][:300]}')
    print()

In [ ]:
# Length stats
df['original_len'] = df['Text'].str.len()
df['cleaned_len'] = df['cleaned_text'].str.len()

avg_orig = df['original_len'].mean()
avg_clean = df['cleaned_len'].mean()
reduction = (1 - avg_clean / avg_orig) * 100

print('Average character length:')
print(f'  Original : {avg_orig:.0f}')
print(f'  Cleaned  : {avg_clean:.0f}')
print(f'  Reduction: {reduction:.1f}%')

## 7. Save the Cleaned Dataset

In [ ]:
# Drop helper length columns before saving
df_out = df.drop(columns=['original_len', 'cleaned_len'])

OUTPUT_PATH = '../data/processed/Reviews_cleaned.csv'
df_out.to_csv(OUTPUT_PATH, index=False)

print(f'Saved cleaned data to {OUTPUT_PATH}')
print(f'Final shape: {df_out.shape}')

## 8. Next Steps

- Apply sentiment scoring with **TextBlob** and **VADER**
- Explore correlation between sentiment and star ratings
- Analyze the `HelpfulnessNumerator` / `HelpfulnessDenominator` columns
- Import processed data into Power BI for dashboard building